In [19]:
from __future__ import absolute_import, division, print_function, unicode_literals
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.optimizers import Adam
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import preprocessing
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
import random

In [20]:
import requests
def download_file(url, filename):
    response = requests.get(url)
    if response.status_code == 200:
        with open(filename, 'wb') as file:
            file.write(response.content)
        print(f"File successfully downloaded as {filename}")
    else:
        print("Failed to retrieve the file. Status code:", response.status_code)

In [21]:
# READ THE INPUT DATA
url = 'https://raw.githubusercontent.com/ageltser1/Oil-OXY/refs/heads/main/OXY_historical_data2.csv'
filename = "OXY_historical_data2.csv"
download_file(url, filename)

File successfully downloaded as OXY_historical_data2.csv


In [22]:
input_filename = filename
df = pd.read_csv(input_filename, dtype='float64', header=0)  # Read CSV while keeping all columns
input_data = df.to_numpy()


print('input_data:', input_data.shape)

print()

# Load your data into a DataFrame
df = pd.read_csv('OXY_historical_data2.csv')

# Check for NaN values
nan_counts = df.isna().sum()

# Display columns with NaN values and their counts
nan_columns = nan_counts[nan_counts > 0]
print(nan_columns)

# Optional: Display rows with NaN values
rows_with_nan = df[df.isna().any(axis=1)]
print(rows_with_nan)

print()


print(df.isna().sum())
print()


# PRINT SOME OF THE INPUT DATA EXAMPLES
print('OXY_Close_PD   OXY_Volume       VIX_Close     OXY_PD_AVGs       Label')
for i in range(0, 19):
    print('{:4.0e}'.format(input_data[i,0]), '      ',
          '{:4.0e}'.format(input_data[i,1]), '         ',
          '{:3.0e}'.format(input_data[i,2]), '          ',
          '{:3.0e}'.format(input_data[i,3]), '             ',
          '{:3.0e}'.format(input_data[i, 4]), '        '
          )

input_data: (3828, 7)

PD_AVGs         16
Label         3580
Unnamed: 6      16
dtype: int64
      OXY_Close_PD  OXY_Volume  VIX_Close   PD_AVGs  Label  OXY_Close  \
0        -0.007220     3515432  19.350000       NaN    NaN  51.145203   
1         0.012000     5578661  19.160000       NaN    NaN  51.758953   
2        -0.007426     4300559  19.059999       NaN    NaN  51.374577   
3         0.010257     4336465  18.129999       NaN    NaN  51.901543   
4        -0.009078     4079384  17.549999       NaN    NaN  51.430378   
...            ...         ...        ...       ...    ...        ...   
3823      0.008252     8347400  21.700001 -0.020134    NaN  47.650002   
3824      0.006506     9812700  19.900000 -0.016602    NaN  47.959999   
3825      0.000000     9062300  19.799999 -0.001956    NaN  47.959999   
3826     -0.000417    23304400  19.280001  0.019957    NaN  47.939999   
3827     -0.000417    10159500  17.480000 -0.080829    NaN  47.919998   

      Unnamed: 6  
0          

In [23]:
"""
training_split = 0.80
seed = 16
train_data, test_data = train_test_split(input_data, train_size=training_split, random_state=seed)
print('train_data:', train_data.shape)
print('test_data:', test_data.shape)
print()
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)
print()
"""

training_size = 3062  # or int(len(input_data) * 0.8)

train_data = input_data[:training_size]       # First 3062 rows
test_data = input_data[training_size:]        # Remaining rows

print('train_data:', train_data.shape)
print('test_data:', test_data.shape)

train_data: (3062, 7)
test_data: (766, 7)


In [24]:
nFeatures = 4
ground_truth_col = 4

train_X = train_data[:, :nFeatures]            # columns 0 to 3 (features)
train_truth = train_data[:, ground_truth_col]  # column 5 (labels)

#train_truth = train_truth-1   # train_truth has values ranging between 1 and 3.  To match the softmax output
                              # module, the Ground Truth array needs to have values ranging between 0 and 2, so
                              # we simply subtract 1 from the values.
print('train_truth shape[0]', train_truth.shape[0])
# for the test data
test_X = test_data[:, :nFeatures]  ## columns 0 to 3 (features)
test_truth = test_data[:, ground_truth_col]   # column 5 (labels)

print('train_X', train_X.shape)
print('train_truth', train_truth.shape)
print('test_X', test_X.shape)
print('test_truth', test_truth.shape)
print()
n_test_examples = test_X.shape[0]


print('OXY_Close_PD    OXY_Volume     VIX_Close     OXY_PD_AVGs          Label')
for i in range(0, 19):
    print('  {:4.0e}'.format(train_X[i,0]), '       ',
          '{:4.0e}'.format(train_X[i,1]), '        ',
          '{:3.0e}'.format(train_X[i,2]), '          ',
          '{:3.0e}'.format(train_X[i,3]), '             ',
          '{:3.0e}'.format(train_truth[i]), '        '
          )

train_truth shape[0] 3062
train_X (3062, 4)
train_truth (3062,)
test_X (766, 4)
test_truth (766,)

OXY_Close_PD    OXY_Volume     VIX_Close     OXY_PD_AVGs          Label
  -7e-03         4e+06          2e+01            nan               nan         
  1e-02         6e+06          2e+01            nan               nan         
  -7e-03         4e+06          2e+01            nan               nan         
  1e-02         4e+06          2e+01            nan               nan         
  -9e-03         4e+06          2e+01            nan               nan         
  -3e-02         7e+06          2e+01            nan               nan         
  -7e-03         8e+06          2e+01            nan               nan         
  8e-03         5e+06          2e+01            nan               nan         
  -2e-02         8e+06          2e+01            nan               nan         
  6e-03         6e+06          2e+01            nan               nan         
  -1e-02         6e+06          2

In [25]:
# SCALE EACH FEATURE TO BE CENTERED ON ZERO, WITH A STANDARD DEVIATION OF 1
scaler_obj = preprocessing.StandardScaler().fit(train_X)   # scaler_obj will scale each Feature (column) independently
train_X_std = scaler_obj.transform(train_X)  # scale each training data Feature (column)
test_X_std = scaler_obj.transform(test_X)  # scale each test data Feature (column)


for i in range(0,3062):
    print(train_X_std[i,0], '      ',
          train_X_std[i,1], '         ',
          train_X_std[i,2], '          ',
        train_X_std[i,3], '             '
         )


print('train_X', train_X_std.shape)
print('train_truth', train_truth.shape)
print('test_X', test_X_std.shape)
print('test_truth', test_truth.shape)


-0.27722015179342946        -0.5039183160608646           0.15585100916964567            nan              
0.42618097179608294        -0.29892604758674046           0.12958053674841769            nan              
-0.28476256065287386        -0.4259119654471985           0.11575395834412473            nan              
0.3623993341507714        -0.4223445221862556           -0.012832775048609359            nan              
-0.3452117916223198        -0.44788682377836986           -0.0930266139956602            nan              
-1.2481806468563827        -0.15058743286394705           0.003759172130401861            nan              
-0.25494479949176785        -0.08959891726571231           -0.05154687878277988            nan              
0.27196079352163954        -0.38448209215320733           -0.08196540436608454            nan              
-0.6195192619015513        -0.033225246110865164           -0.04325103654527028            nan              
0.19105070607311828        -0.2

In [26]:
# DISCUSSION #5:  The DNN Model

n_possible_outcomes = 3
model = keras.models.Sequential()
model.add(keras.layers.InputLayer(input_shape=[nFeatures,]))               # Input Layer
model.add(keras.layers.Dense(nFeatures, activation='relu'))                # Hidden Layer 1
model.add(keras.layers.Dense(nFeatures, activation='relu'))                # Hidden Layer 2
model.add(keras.layers.Dense(n_possible_outcomes, activation='softmax'))   # Output Layer

model.summary()
model.compile(loss='sparse_categorical_crossentropy',optimizer=Adam(0.1), metrics=["accuracy"])


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 4)              │            20 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 4)              │            20 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 3)              │            15 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 55 (220.00 B)

 Trainable params: 55 (220.00 B)

 Non-trainable params: 0 (0.00 B)

In [27]:
# :::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::
# DISCUSSION #6:  Run the model
# -----------------------------------------------------------------------------------------------------------------
# The TensorFlow 'fit' method is used to compute the best weights for the Training Examples.  The 136 weights
# are trained over 10 epochs.  Note that the softmax Output Layer outputs a probability for each of the possible
# outcomes.
# Each probability will be a number between 0 and 1.  For each Training Example, the largest of the 3 probabilites
# is chosen as the 'answer' to be compared against the Ground Truth value.  The comparison of the Training Examples
# to their Ground Truth values are stored in the 'history' object for plotting.
#
# The Test Examples are also input to the fit method and the predictions from our new model (derived from the
# Training Examples) are compared against the matching Test Ground Truth values.  This gives a validation
# measure of how the model will do when data it has never seen before are input.  The comparison of the Test
# Examples to their Ground Truth values are also stored in the 'history' object for plotting.

# New Running Sum = ((Current Running Sum +1)*(New Gain + 1)) - 1
# :::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::
nEpochs = 100
history = model.fit(train_X_std, train_truth, batch_size=16, epochs=nEpochs, validation_data=(test_X_std,test_truth))


Epoch 1/100


InvalidArgumentError: Graph execution error:

Detected at node compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/SparseSoftmaxCrossEntropyWithLogits defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/usr/local/lib/python3.11/dist-packages/colab_kernel_launcher.py", line 37, in <module>

  File "/usr/local/lib/python3.11/dist-packages/traitlets/config/application.py", line 992, in launch_instance

  File "/usr/local/lib/python3.11/dist-packages/ipykernel/kernelapp.py", line 712, in start

  File "/usr/local/lib/python3.11/dist-packages/tornado/platform/asyncio.py", line 205, in start

  File "/usr/lib/python3.11/asyncio/base_events.py", line 608, in run_forever

  File "/usr/lib/python3.11/asyncio/base_events.py", line 1936, in _run_once

  File "/usr/lib/python3.11/asyncio/events.py", line 84, in _run

  File "/usr/local/lib/python3.11/dist-packages/ipyker

In [ ]:
# DISCUSSION #7:  Print/plot information about the model's accuracy
# PRINT THE ACCURACY AND LOSS
train_loss,train_accuracy = model.evaluate(train_X_std, train_truth, batch_size=16, verbose=0)
print()
print("Training Accuracy = {:.2f}".format(train_accuracy))
print("Training Loss     = {:.2f}".format(train_loss))
test_loss,test_accuracy = model.evaluate(test_X_std, test_truth, batch_size=16, verbose=0)
print("Test Accuracy     = {:.2f}".format(test_accuracy))
print("Test Loss         = {:.2f}".format(test_loss))

# PLOT THE ACCURACY AND LOSS
pd.DataFrame(history.history).plot(figsize=(8,5))
plt.grid(True)
plt.gca().set_ylim(0,1)
plt.suptitle("Loss and Accuracy for " + str(nEpochs) + " epochs")
plt.title("(Note that 'val' = validation test scores)")
plt.show()


In [ ]:
# :::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::
# DISCUSSION #8:  Print a Confusion Matrix on the Test Examples for the 3rd Class Passengers
# :::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::
# PRINT A CONFUSION MATRIX FOR THE TEST EXAMPLES, FOR JUST 3RD CLASS
probabilities = model.predict(test_X_std)  # probabilities = 2D array (rows=examples, cols=the 3 outcome probabilities)
predicted_Action = tf.argmax(probabilities, axis=1).numpy()   # predictions is a 1D array of the 'winning' class (rows=examples)

# Remap predicted_class and test_truth so we're only dealing with 3rd class tickets.  This means we will set all 1st and
# 2nd class tickets to False (==0) and the 3rd class tickets to True (==1)
nPredictions = predicted_Action.shape[0]
predicted_HOLD = np.zeros(nPredictions)    # Preset everything to zero (False)
test_truth_for_HOLD = np.zeros(nPredictions)     # Preset everything to zero (False)
for i in range (nPredictions):
      if predicted_Action[i] == 2:  predicted_HOLD[i] = 2   # Reset this example to True
      if test_truth[i] == 1:     test_truth_for_HOLD[i] = 1    # Reset this example to True

# the confusion matrix on the remapped values for 3rd class tickets
((n_true_negatives, n_false_positives), (n_false_negatives, n_true_positives)) \
      = confusion_matrix(test_truth_for_HOLD, predicted_HOLD)
print()
print()
print('CONFUSION MATRIX for Predicting 3rd Class Tickets.  Passenger counts for', n_test_examples, 'Test Examples')
print()
print('         Correctly Predicted is NOT 3rd Class Ticket:      ', n_true_negatives, '  |  ', n_false_negatives, ' :Predicted is NOT 3rd Class Ticket, but actually was')
print('                                         ------------------------------------------------')
print('         Predicted 3rd Class Ticket, but actually was NOT:  ', n_false_positives, '  | ', n_true_positives, ' :Correctly Predicted is 3rd Class Ticket')
print()


In [ ]:
# :::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::
# DISCUSSION #9:  Print some examples
# -----------------------------------------------------------------------------------------------------------------
# Here I've printed out the first 20 Test Examples, the 3 resulting probabilities, and the resulting prediction,
# and the actual Ground Truth outcome. The predictions should be correct about 83% of the time.
#
# Note that the model, itself, is relatively opaque.  It can predict the ticket class of an individual, but it's
# difficult to know why!  And this is a simple model with only 120 weights.  It would be nearly impossible to translate
# those weights into an understanding of the relative importance of each input Feature.  Therefore, the model is rather
# like a black box.  Sadly, this opacity is typical of AI models.
# :::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::::
# PRINT SOME OF THE INPUT DATA EXAMPLES
print()
print()


# PRINT SOME OF THE INPUT DATA EXAMPLES
print('OXY_Close_PD   OXY_Volume       VIX_Close     OXY_PD_AVGs       Label   ||  Hold(0)   Sell(1)   Buy(2)           Actual_AVG_Label'')
for i in range(0, 19):
    print('{:4.0e}'.format(input_data[i,0]), '      ',
          '{:4.0e}'.format(input_data[i,1]), '         ',
          '{:3.0e}'.format(input_data[i,2]), '          ',
          '{:3.0e}'.format(input_data[i,3]), '             ',
          '{:3.0e}'.format(input_data[i, 4]), '        '
           '{:4.2f}'.format(probabilities[i,0]), '  ',
          '{:4.2f}'.format(probabilities[i,1]), '  ',
          '{:4.2f}'.format(probabilities[i,2]), '    =  ',
          '{:3.0f}'.format(int(test_truth[i]))
          )


"""
print('OXY_Open   OXY_High       OXY_Low     OXY_Adj_Close       OXY_Volume       OIL_Open    OIL_Close    OIL_High     OIL_Low   AVG_Label ||  Hold(0)   Sell(1)   Buy(2)           Actual_AVG_Label')
for i in range(0, 200):
    print('{:4.0f}'.format(input_data[i,0]), '      ',
          '{:4.0f}'.format(input_data[i,1]), '         ',
          '{:3.0f}'.format(input_data[i,2]), '          ',
          '{:3.0f}'.format(input_data[i,3]), '             ',
          '{:3.0f}'.format(input_data[i, 4]), '        ',
          '{:3.0f}'.format(input_data[i, 5]), '       ',
          '{:3.0f}'.format(input_data[i, 6]), '         '
          '{:3.0f}'.format(input_data[i,7]), '        ',
          '{:3.0f}'.format(input_data[i,8]), '       ',
          '{:3.0f}'.format(input_data[i, 9]), '         ',
          '{:4.2f}'.format(probabilities[i,0]), '  ',
          '{:4.2f}'.format(probabilities[i,1]), '  ',
          '{:4.2f}'.format(probabilities[i,2]), '    =  ',
          '{:3.0f}'.format(predicted_HOLD[i]), '        ',
          '{:3.0f}'.format(int(test_truth[i]))
          )

"""